# Putting the G(enerate) in RAG

In [ ]:
!pip install supabase
!pip install pinecone
from supabase import create_client, Client
import datetime
from pinecone import Pinecone, ServerlessSpec

from typing import Dict, Optional, Any
import os
from openai import OpenAI
import pandas as pd

from pydantic import BaseModel, Field
from typing import List, Dict, Tuple

# Setting up our environment

In [ ]:
url: str = os.environ.get('SUPABASE_URL')
key: str = os.environ.get('SUPABASE_KEY')
supabase: Client = create_client(url, key)

In [ ]:
# Initialize the OpenAI client with the API key from user data
client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

In [ ]:
pinecone_key = os.environ.get('PINECONE_API_KEY')
INDEX_NAME = 'semantic-search-rag'
ENGINE = 'text-embedding-3-small'
NAMESPACE = 'default'

pc = Pinecone(
    api_key=pinecone_key
)

# helper functions to get lists of embeddings from the OpenAI API
def get_embedding(text, engine=ENGINE):
    response = client.embeddings.create(
        input=[text],
        model=engine
    )
    return response.data[0].embedding

len(get_embedding('hi'))

In [ ]:
# Store the index as a variable
index = pc.Index(name=INDEX_NAME)
index

In [ ]:
def query_from_pinecone(query, top_k=1, include_metadata=True):
    # get embedding from THE SAME embedder as the documents
    query_embedding = get_embedding(query, engine=ENGINE)

    return index.query(
      vector=query_embedding,
      top_k=top_k,
      namespace=NAMESPACE,
      include_metadata=include_metadata   # gets the metadata (dates, text, etc)
    ).get('matches')

len(query_from_pinecone('I lost my card'))

In [ ]:
query_from_pinecone('I lost my card')

In [ ]:
FINAL_ANSWER_TOKEN = 'REDACTED'
STOP = '[END]'
PROMPT_TEMPLATE = """Today is {today} and you can retrieve information from a database. Respond to the user's input as best as you can.

Here is an example of the conversation format:

[START]
User Input: the input question you must answer
Context: retrieved context from the database
Context URL: context url
Context Score : a score from 0 - 1 of how strong the information is a match
Assistant Thought: This context has sufficient information to answer the question.
Assistant Response: your final answer to the original input question which could be I don't have sufficient information to answer the question.
[END]
[START]
User Input: another input question you must answer
Context: more retrieved context from the database
Context URL: context url
Context Score : another score from 0 - 1 of how strong the information is a match
Assistant Thought: This context does not have sufficient information to answer the question.
Assistant Response: your final answer to the second input question which could be I don't have sufficient information to answer the question.
[END]
[START]
User Input: another input question you must answer
Context: more retrieved context from the database
Context URL: context url
Context Score : another score from 0 - 1 of how strong the information is a match
Assistant Thought: A previous piece of context has the answer to this question
Assistant Response: your final answer to the second input question which could be I don't have sufficient information to answer the question.
[END]
[START]
User Input: another input question you must answer
Context: NO CONTEXT FOUND
Context URL: NONE
Context Score : 0
Assistant Thought: We either could not find something or we don't need to look something up
Assistant Response: I'm sorry I don't know.
[END]

Begin:

{running_convo}
"""

class RagBot(BaseModel):
    llm: Any
    prompt_template: str = PROMPT_TEMPLATE
    stop_pattern: List[str] = [STOP]
    user_inputs: List[str] = []
    ai_responses: List[str] = []
    contexts: List[Tuple[str, float]] = []
    verbose: bool = False
    threshold: float = 0.6

    def query_from_pinecone(self, query, top_k=1, include_metadata=True):
        return query_from_pinecone(query, top_k, include_metadata)

    @property
    def running_convo(self):
        convo = ''
        for index in range(len(self.user_inputs)):
            convo += f'[START]\nUser Input: {self.user_inputs[index]}\n'
            convo += f'Context: {self.contexts[index][0]}\nContext URL: {self.contexts[index][1]}\nContext Score: {self.contexts[index][2]}\n'
            if len(self.ai_responses) > index:
                convo += self.ai_responses[index]
                convo += '\n[END]\n'
        return convo.strip()

    def run(self, question: str):
        self.user_inputs.append(question)
        top_response = self.query_from_pinecone(question)[0]
        if self.verbose:
            print(top_response['score'])
        if top_response['score'] >= self.threshold:
            self.contexts.append(
                (top_response['metadata']['text'], top_response['metadata']['url'], top_response['score']))
        else:
            self.contexts.append(('NO CONTEXT FOUND', 'NONE', 0))

        prompt = self.prompt_template.format(  # behold, the augmentation
                today = datetime.date.today(),
                running_convo=self.running_convo
        )
        if self.verbose:
            print('--------')
            print('PROMPT')
            print('--------')
            print(prompt)
            print('--------')
            print('END PROMPT')
            print('--------')
        generated = self.llm.generate(prompt, stop=self.stop_pattern)
        if self.verbose:
            print('--------')
            print('GENERATED')
            print('--------')
            print(generated)
            print('--------')
            print('END GENERATED')
            print('--------')
        self.ai_responses.append(generated)
        if FINAL_ANSWER_TOKEN in generated:
            generated = generated.split(FINAL_ANSWER_TOKEN)[-1]
        return generated

# Using OpenAI as our Generator

In [ ]:

# Define a class for the Chat Language Model
class OpenAIChatLLM(BaseModel):
    model: str = 'gpt-4o'  # Default model to use
    temperature: float = 0.0  # Default temperature for generating responses

    # Method to generate a response from the model based on the provided prompt
    def generate(self, prompt: str, stop: List[str] = None):
        # Create a completion request to the OpenAI API with the given parameters
        response = client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=self.temperature,
            stop=stop
        )

        # Insert the details of the prompt and response into the 'cost_projecting' table in Supabase
        supabase.table('cost_projecting').insert({
            'prompt': prompt,
            'response': response.choices[0].message.content,
            'input_tokens': response.usage.prompt_tokens,
            'output_tokens': response.usage.completion_tokens,
            'model': self.model,
            'inference_params': {
                'temperature': self.temperature,
                'stop': stop
            },
            'is_openai': True,
            'app': 'RAG'
        }).execute()

        # Return the generated response content
        return response.choices[0].message.content


In [ ]:
c = OpenAIChatLLM()
c.generate('hi')

In [ ]:
openai_rag = RagBot(llm=OpenAIChatLLM(temperature=0, model='gpt-4o-mini'), stop_pattern=['[END]'], verbose=False)
print(openai_rag.run('I lost my medicare card'))

In [ ]:
print(openai_rag.running_convo)

In [ ]:
print(openai_rag.run('What was that number again?'))

In [ ]:
print(openai_rag.running_convo)

In [ ]:
print(openai_rag.run('amazing, thanks!'))

In [ ]:
print(openai_rag.running_convo)

In [ ]:
response = supabase.table('cost_projecting').select("*").eq('app', 'RAG').execute()
completions_df = pd.DataFrame(response.data)
completions_df.index = pd.to_datetime(completions_df['created_at'])

completions_df.tail()

In [ ]:
prices = { # per 1M tokens
    'gpt-3.5-turbo': {
        'prompt': 0.5,
        'completion': 1.5
    },
    'gpt-4o': {
        'prompt': 5,
        'completion': 15
    },
}

def calculate_cost(input_tokens, output_tokens, model):
    if model not in prices:
        return None

    prompt_cost = input_tokens / 1e6
    completion_cost = output_tokens / 1e6

    return prompt_cost + completion_cost

calculate_cost(354, 400, 'gpt-3.5-turbo'), calculate_cost(354, 400, 'gpt-4o')

In [ ]:
# run calculate_cost over every row
completions_df['cost'] = completions_df.apply(
    lambda row: calculate_cost(row['input_tokens'], row['output_tokens'], row['model']), axis=1
    )

In [ ]:
completions_df['cost'].resample('W-Mon').sum().sort_index()

In [ ]:
completions_df['cost'].resample('W-Mon').sum().sort_index().plot(kind='bar')

# Using Claude as our Generator

In [ ]:
from anthropic import Anthropic
anthropic_client = Anthropic(
    api_key=os.environ.get("ANTHROPIC_API_KEY"),
)

In [ ]:
def test_prompt_anthropic(prompt, suppress=False, model='claude-3-opus-20240229', **kwargs):
    " a simple function to take in a prompt and run it through a given non-chat model "

    message = anthropic_client.messages.create(
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        model=model,
        max_tokens=1024,
        **kwargs
    )
    answer = message.content[0].text
    if not suppress:
        print(f'PROMPT:\n------\n{prompt}\n------\nRESPONSE\n------\n{answer}')
    else:
        return answer



In [ ]:
test_prompt_anthropic('1+1=', model='claude-3-5-haiku-latest')

In [ ]:
# Define a class for the Chat Language Model
class AnthropicChatLLM(BaseModel):
    model: str = 'claude-3-5-haiku-latest'  # Default model to use

    # Method to generate a response from the model based on the provided prompt
    def generate(self, prompt: str, stop: List[str] = None):
        # Create a completion request to the Anthropic API with the given parameters
        response = test_prompt_anthropic(prompt, model=self.model, suppress=True)

        # no supabase here, feel free to add it

        # Return the generated response content
        return response


In [ ]:
anthropic_llm  = AnthropicChatLLM()

anthropic_llm.generate('What is 1+1?')

In [ ]:
anthropic_rag = RagBot(llm=AnthropicChatLLM(), stop_pattern=['[END]'])
print(anthropic_rag.run('I lost my medicare card'))

In [ ]:
print(anthropic_rag.running_convo)

# Using Llama-3 8b as our Generator

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")

In [ ]:
import requests

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>"),
    tokenizer.convert_tokens_to_ids("assistant"),
]

def test_prompt_llama_3_8b(prompt, suppress=False, **kwargs):

    API_URL = "https://my03m9749ssz7t6h.us-east-1.aws.endpoints.huggingface.cloud"
    headers = {
    	"Accept" : "application/json",
    	"Authorization": f"Bearer {userdata.get('HF_TOKEN')}",
    	"Content-Type": "application/json"
    }

    llama_prompt = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

    def query(payload):
    	response = requests.post(API_URL, headers=headers, json=payload)
    	return response.json()

    kwargs["return_text"] = False
    kwargs["return_full_text"] = False
    kwargs['max_new_tokens'] = 512
    kwargs['stop'] = ["<|end_of_text|>", "<|eot_id|>"]

    output = query({
    	"inputs": llama_prompt,
    	"parameters": kwargs
    })
    answer = output[0]['generated_text']
    if not suppress:
        print(f'PROMPT:\n------\n{llama_prompt}\n------\nRESPONSE\n------\n{answer}')
    else:
        return answer

test_prompt_llama_3_8b('1+1=?')

In [ ]:
test_prompt_llama_3_8b('1+1=?', suppress=True)

In [ ]:
class LlamaChatLLM(BaseModel):
    temperature: float = 0.3
    do_sample: bool = True
    max_new_tokens: int = 256

    def generate(self, prompt: str, stop: List[str] = None):
        response = test_prompt_llama_3_8b(prompt, suppress=True)
        return response

In [ ]:
llama_rag = RagBot(llm=llama_llm, verbose=False, stop_pattern=['[END]'])
print(llama_rag.run('I lost my medicare card'))

In [ ]:
llama_rag.user_inputs

In [ ]:
llama_rag.ai_responses

In [ ]:
print("[START]\nUser Input: I lost my medicare card\nContext: If your Medicare card was lost, stolen, or destroyed, you can request a replacement online at Medicare.gov. You can print an official copy of your card from your online Medicare account or call 1-800-MEDICARE (1-800-633-4227 TTY 1-877-486-2048) to order a replacement card to be sent in the mail.\nContext URL: https://faq.ssa.gov/en-us/Topic/article/KA-01735\nContext Score: 0.646264791\nAssistant Thought: This context has sufficient information to answer the question.\nAssistant Response: If you've lost your Medicare card, you can request a replacement online at Medicare.gov or call 1-800-MEDICARE to order a new card to be sent in the mail.")

In [ ]:
llama_rag.contexts

In [ ]:
# https://huggingface.co/CohereForAI/c4ai-command-r-v01-4bit

# Using [Ollama](https://ollama.com/) to use Llama locally

In [ ]:
import ollama

class OllamaLLM(BaseModel):
    model_name:str = "llama3.2"

    def generate(self, prompt: str, stop: List[str] = None):
        messages = [{'role': 'user', 'content': prompt}]
        response = ollama.chat(model=self.model_name, messages=messages, options={'stop': stop})
        return response['message']['content']

In [ ]:
o_llama = OllamaLLM()
o_llama.generate('What is 1+1?')

In [ ]:
# removed stop sequence because this smaller model and it repeated info back to me. I'd adjust the prompt if we wanted to use this model
# The model hallucinated a brand new question!
o_llama_rag = RagBot(llm=o_llama, verbose=False, stop_pattern=[])
print(o_llama_rag.run('I lost my medicare card'))

In [ ]:
o_llama_rag.ai_responses

In [ ]:
print(o_llama_rag.running_convo)

# Using Gemini as our Generator

In [ ]:
import google.generativeai as genai

In [ ]:
class GeminiLLM(BaseModel):
    api_key:str = os.getenv("GEMINI_API_KEY")
    model_name:str = 'gemini-1.5-flash'


    def generate(self, prompt, max_output_tokens=1024, stop=None, **kwargs):
        genai.configure(api_key=self.api_key)
        client = genai.GenerativeModel(self.model_name)

        # Convert messages to the format expected by the SDK
        history = []

        # Start a chat session with the existing history
        chat = client.start_chat(history=history)

        # Prepare the generation configuration
        generation_config = genai.types.GenerationConfig(
            max_output_tokens=max_output_tokens,
            stop_sequences=stop,
            **kwargs
        )

        # Send the final user message and get the response
        response = chat.send_message(
            prompt,
            generation_config=generation_config
        )
        return response.candidates[0].content.parts[0].text.strip()


In [ ]:
gemini_llm = GeminiLLM()
gemini_llm.generate('1+1?')

In [ ]:
# This model also repeated my prompt back to me. I'd adjust the prompt if we wanted to use this model but it technically works!
gemini_llm_rag = RagBot(llm=gemini_llm, verbose=False, stop_pattern=['[END]'])
print(gemini_llm_rag.run('I lost my medicare card'))

In [ ]:
print(gemini_llm_rag.running_convo)

# Using [Command-R](https://cohere.com/blog/command-r?ref=cohere-ai.ghost.io) as our Generator

In [ ]:
!pip install bitsandbytes accelerate torch[transformers]

In [ ]:
from transformers import AutoTokenizer

model_id = "CohereForAI/c4ai-command-r-v01"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# define conversation input:
conversation = [
    {"role": "user", "content": "Whats the biggest penguin in the world?"}
]
# define documents to ground on:
documents = [
    { "title": "Tall penguins", "text": "Emperor penguins are the tallest growing up to 122 cm in height." },
    { "title": "Penguin habitats", "text": "Emperor penguins only live in Antarctica."}
]

# render the tool use prompt as a string:
grounded_generation_prompt = tokenizer.apply_grounded_generation_template(
    conversation,
    documents=documents,
    citation_mode="accurate", # or "fast"
    tokenize=False,
    add_generation_prompt=True,
)
print(grounded_generation_prompt)


In [ ]:
# render the tool use prompt as a string:
grounded_generation_tokens = tokenizer.apply_grounded_generation_template(
    conversation,
    documents=documents,
    citation_mode="accurate", # or "fast"
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)
print(grounded_generation_tokens.shape)


In [ ]:
# pip install 'transformers>=4.39.1' bitsandbytes accelerate
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "CohereForAI/c4ai-command-r-v01-4bit"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

In [ ]:
gen_tokens = model.generate(
    grounded_generation_tokens,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.3,
    )

gen_text = tokenizer.decode(gen_tokens[0])
print(gen_text)


In [ ]:
def format_for_command_r(documents):
    return [{'title': f'Document {index + 1}', 'text': document['metadata']['text']} for index, document in enumerate(documents)]

command_r_docs = format_for_command_r(query_from_pinecone('I lost my card', top_k=3, include_metadata=True))

len(command_r_docs)

In [ ]:
# define conversation input:
conversation = [
    {"role": "user", "content": "I lost my card"}
]

# render the tool use prompt as a string:
grounded_generation_tokens = tokenizer.apply_grounded_generation_template(
    conversation,
    documents=command_r_docs,
    citation_mode="citation",
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

gen_tokens = model.generate(
    grounded_generation_tokens,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.3,
    )

gen_text = tokenizer.decode(gen_tokens[0])
print(gen_text)
